# Bóc tách não bộ XGBoost

Mô hình của chúng ta sử dụng **Gradient Boosting** với **156 cây quyết định**. Trong đó, mỗi cây chỉ đóng góp một phần nhỏ (Learning rate = 0.05) và liên tục sửa sai cho cây đi trước.

Mỗi lần chạy lại  `2. Tính toán và Xuất Sơ đồ Cây bất kỳ` Notebook này, hệ thống sẽ **Bốc thăm ngẫu nhiên 1 Cây quyết định bất kỳ** để dự đoán rủi ro cho cùng 1 Khách hàng.

Từ đó, sẽ thấy rõ rằng dù từng cây có góc nhìn khác nhau (cộng hay trừ điểm), nhưng kết quả Tổng hợp cuối cùng là không bao giờ đổi!

### 1. Tải Mô hình và chọn 1 khách hàng cố định

In [ ]:
import sys
import os
import joblib
import random
import datetime
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display, HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

# Trỏ đường dẫn về thư mục gốc DATN để import được source code
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from services.ml_engine.src.machine_learning.config import get_task_config
from services.ml_engine.src.machine_learning.preprocessing import DataPreprocessor
from services.ml_engine.src.machine_learning.features import FeatureEngineer

# Tải cấu hình và mô hình
config = get_task_config("credit_risk")
model_path = "../services/ml_engine/models/credit_risk_model.joblib"
model = joblib.load(model_path)
booster = model.get_booster()
n_trees = len(booster.get_dump())

# Chuẩn bị dữ liệu
preprocessor = DataPreprocessor(config)
df = preprocessor.clean(save=False)
X_train, X_test, y_train, y_test = preprocessor.split_data(df)

feature_eng = FeatureEngineer(config)
feature_eng.build_encoder(X_train, y_train)
X_test_enc = feature_eng.transform(X_test)

# ==================================================================
# CHỌN NGẪU NHIÊN 1 KHÁCH HÀNG TỪ TẬP TEST
# ==================================================================
sample_idx = random.randint(0, len(X_test_enc) - 1)

sample_customer = X_test_enc.iloc[[sample_idx]]
actual_label = "VỠ NỢ (Bad)" if y_test.iloc[sample_idx] == 1 else "TRẢ TỐT (Good)"

print("="*50)
print(f" HỒ SƠ KHÁCH HÀNG SỐ: {sample_idx}")
print(f" Thực tế khách hàng này là: {actual_label}")
print("="*50)
display(sample_customer)

### 2. Tính toán và Xuất Sơ đồ Cây bất kỳ

In [ ]:
# Bốc thăm 1 cây bất kỳ (Từ 0 đến 155)
random_tree_idx = random.randint(0, n_trees - 1)

dmat = xgb.DMatrix(sample_customer)

# --- BƯỚC 2.1: TÍNH TOÁN ĐIỂM SỐ ---
base_margin = booster.predict(dmat, iteration_range=(0, 0), output_margin=True)[0]
val_before = booster.predict(dmat, iteration_range=(0, random_tree_idx), output_margin=True)[0]
val_after = booster.predict(dmat, iteration_range=(0, random_tree_idx + 1), output_margin=True)[0]
single_tree_contrib = val_after - val_before
total_margin = booster.predict(dmat, output_margin=True)[0]

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

final_prob = sigmoid(total_margin)
predict_label = "VỠ NỢ (Bad)" if final_prob > 0.5 else "TRẢ TỐT (Good)"

# --- BƯỚC 2.2: LẤY THÔNG TIN CÂY VÀ TRUY VẾT ---
tree_dump = booster.get_dump(with_stats=True)[random_tree_idx]
lines = [line.strip() for line in tree_dump.strip().split('\n') if line.strip()]
n_nodes = len(lines)
n_leaves = len([l for l in lines if "leaf=" in l])

# Tính toán thống kê chuyên sâu (Depth, Cover, Gain)
tree_edges = {}
gains = []
covers = []
leaf_weights = []
for line in lines:
    if "gain=" in line:
        gains.append(float(line.split("gain=")[1].split(",")[0]))
    if "cover=" in line:
        covers.append(float(line.split("cover=")[1].split(",")[0]))
    if "leaf=" in line:
        leaf_weights.append(float(line.split("leaf=")[1].split(",")[0]))
    if "yes=" in line:
        node_id = line.split(":")[0]
        yes_id = line.split("yes=")[1].split(",")[0]
        no_id = line.split("no=")[1].split(",")[0]
        tree_edges[node_id] = (yes_id, no_id)

def get_depth(node):
    if node not in tree_edges:
        return 0
    left, right = tree_edges[node]
    return 1 + max(get_depth(left), get_depth(right))

actual_depth = get_depth('0')
max_gain = max(gains) if gains else 0
min_cover = min(covers) if covers else 0
root_feature = lines[0].split("[")[1].split("<")[0] if len(lines)>0 and "[" in lines[0] else "N/A"
avg_leaf = sum(abs(w) for w in leaf_weights) / len(leaf_weights) if leaf_weights else 0

leaf_indices = booster.predict(dmat, pred_leaf=True)[0]
target_leaf = str(int(leaf_indices[random_tree_idx]))

parents = {}
for line in lines:
    if "yes=" in line:
        node_id = line.split(":")[0]
        yes_id = line.split("yes=")[1].split(",")[0]
        no_id = line.split("no=")[1].split(",")[0]
        parents[yes_id] = (node_id, "yes")
        parents[no_id] = (node_id, "no")

path_edges = []
curr = target_leaf
while curr in parents:
    parent_id, direction = parents[curr]
    path_edges.append((parent_id, curr, direction))
    curr = parent_id

# --- BƯỚC 2.3: XUẤT HTML REPORT ---
html_out = f"""
<div style="background-color:#1e1e1e; color:#ffffff; padding:25px; border-radius:12px; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; box-shadow: 0 6px 12px rgba(0,0,0,0.3); border: 1px solid #444;">
    <h2 style="color:#2ed573; margin-top:0; border-bottom: 2px solid #333; padding-bottom: 10px; font-size: 22px;">
        THÔNG SỐ KỸ THUẬT - CÂY QUYẾT ĐỊNH SỐ {random_tree_idx}
    </h2>
    
    <h3 style="color:#dfe4ea; margin-bottom: 10px; font-size: 18px;">1. THÔNG SỐ CẤU TRÚC & THỐNG KÊ CÂY</h3>
    <table style="width: 100%; text-align: left; border-collapse: collapse; margin-bottom: 20px; font-size: 15px;">
        <tr style="background-color:#2f3542;">
            <td style="padding: 10px; border: 1px solid #444;"><b>Độ sâu thực tế (Depth):</b> <span style="color:#ffa502; font-weight:bold;">{actual_depth}</span></td>
            <td style="padding: 10px; border: 1px solid #444;"><b>Biến ở Gốc (Root):</b> {root_feature}</td>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #444;"><b>Số nút rẽ nhánh (Nodes):</b> {n_nodes - n_leaves}</td>
            <td style="padding: 10px; border: 1px solid #444;"><b>Số lá kết luận (Leaves):</b> {n_leaves}</td>
        </tr>
        <tr style="background-color:#2f3542;">
            <td style="padding: 10px; border: 1px solid #444;"><b>Lợi ích Cắt nhánh lớn nhất (Max Gain):</b> {max_gain:.4f}</td>
            <td style="padding: 10px; border: 1px solid #444;"><b>Lá mỏng nhất (Min Cover):</b> <span style="color:#ff6b81; font-weight:bold;">{min_cover}</span></td>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #444;"><b>Lực đẩy trung bình của Lá:</b> {avg_leaf:.4f}</td>
            <td style="padding: 10px; border: 1px solid #444;"><b>Sức mạnh Cây {random_tree_idx} đóng góp:</b> <span style="color:#ff6b81; font-weight:bold;">{single_tree_contrib:.4f}</span></td>
        </tr>
    </table>
    
    <h3 style="color:#dfe4ea; margin-bottom: 10px; font-size: 18px;">2. KẾT QUẢ DỰ ĐOÁN (HỆ THỐNG 156 CÂY)</h3>
    <div style="background-color:#2f3542; padding: 15px; border-radius: 8px; font-size: 16px;">
        <p style="margin: 5px 0;">Điểm Margin tích lũy (Total Raw Margin): <b>{total_margin:.4f}</b></p>
        <p style="margin: 5px 0;">Xác suất Vỡ nợ (PD): <b style="color:#ffa502;">{final_prob:.2%}</b></p>
        <p style="margin: 5px 0;">Quyết định của hệ thống: <b style="color:{'#ff4757' if final_prob>0.5 else '#2ed573'}; font-size: 20px;">{predict_label}</b></p>
    </div>
</div>
"""
display(HTML(html_out))

# --- BƯỚC 2.4: TẠO FILE MERMAID ---
mmd_content = "graph TD\n"
link_idx = 0
highlight_styles = ""

for line in lines:
    if "yes=" in line:
        node_id = line.split(":")[0]
        condition = line.split("[")[1].split("]")[0]
        yes_id = line.split("yes=")[1].split(",")[0]
        no_id = line.split("no=")[1].split(",")[0]
        
        mmd_content += f'    N{node_id}["{condition}"]\n'
        
        mmd_content += f'    N{node_id} -->|yes| N{yes_id}\n'
        if (node_id, yes_id, "yes") in path_edges:
            highlight_styles += f'    linkStyle {link_idx} stroke:#059669,stroke-width:5px;\n'
            highlight_styles += f'    style N{node_id} stroke:#059669,stroke-width:5px\n'
        link_idx += 1
        
        mmd_content += f'    N{node_id} -->|no| N{no_id}\n'
        if (node_id, no_id, "no") in path_edges:
            highlight_styles += f'    linkStyle {link_idx} stroke:#059669,stroke-width:5px;\n'
            highlight_styles += f'    style N{node_id} stroke:#059669,stroke-width:5px\n'
        link_idx += 1
        
    elif "leaf=" in line:
        node_id = line.split(":")[0]
        leaf_val = line.split("leaf=")[1]
        mmd_content += f'    N{node_id}(("Leaf: {leaf_val}"))\n'
        if node_id == target_leaf:
            mmd_content += f'    style N{node_id} fill:#10b981,stroke:#047857,stroke-width:5px,color:#ffffff\n'
        else:
            mmd_content += f'    style N{node_id} fill:#fca5a5,stroke:#b91c1c,stroke-width:1px\n'

mmd_content += "\n    %% --- HIGHLIGHT DONG CHAY DU LIEU ---\n"
mmd_content += highlight_styles

timestamp = datetime.datetime.now().strftime("%H%M%S")
mmd_filepath = f"random_tree_{random_tree_idx}.mmd"

with open(mmd_filepath, "w", encoding="utf-8") as f:
    f.write(mmd_content)